In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

Imprting libraries


In [0]:
from datetime import datetime,timedelta
from pyspark.sql import functions as F
import json

In [0]:
from my_schemas import USER_RECENT_PLAYED_SCHEMA

In [0]:
## NORMALIZING  PARAMETER DATE
#  Try to avoid inconsistencies in date format especially for back fill runs 

def normalize_date(run_date:str)->str:
    if "/" in run_date:
        run_date = run_date.replace("/","-")
    
    check_list = run_date.split("-")
    
    if len(check_list[-1]) == 4:
        run_date = check_list[-1] + "-" + check_list[1] + "-" + check_list[0]
    
    return run_date


In [0]:

run_date = dbutils.widgets.get("run_date")
if run_date == "":
    run_date = datetime.now().strftime("%Y-%m-%d")
else:
    run_date = normalize_date(run_date)

In [0]:
run_date

In [0]:
#------ READING ROW DATA ---------
# run_date = datetime.now().strftime("%Y-%m-%d")
df = spark.read.option('multiline','true').json(f's3://my-raw-spotify-data/bronze/user_recent_played/{run_date}/{run_date}_recent_tracks.json')


In [0]:
df = df.withColumn('items',F.explode('items')).select('items')


In [0]:
df_flat = (
    df.select(
        F.from_utc_timestamp(
            F.col('items.played_at').cast('timestamp'), 
            "Europe/Rome"
        ).alias('played_at'),
        F.col('items.track.id').alias('track_id'),
        F.col('items.track.name').alias('track_name'),
        F.col('items.track.artists')[0].getField('id').alias('first_artist_id'),
        F.col('items.track.album.id').alias('album_id'),
    )
)

In [0]:
df_flat = (
    df_flat
    .withColumn(
        "time_of_day",
        F.when(F.hour(F.col("played_at")).between(6, 11), "morning")
         .when(F.hour(F.col("played_at")).between(12, 17), "afternoon")
         .when(F.hour(F.col("played_at")).between(18, 23), "evening")
         .otherwise("night")
    )
    .withColumn('ingested_ts', F.current_timestamp())
)

In [0]:
## CHECKING EXPECTED SCHEMA BEFORE WRITING

assert df_flat.schema == USER_RECENT_PLAYED_SCHEMA,"SCHEMA MISMATCH"


In [0]:
%sql

USE CATALOG my_spotify;
CREATE SCHEMA IF NOT EXISTS silver;


In [0]:
   # Idempotent merge
   
   from delta.tables import DeltaTable

   silver_path = 's3://my-spotify-delta-lakehouse/silver'

   if not DeltaTable.isDeltaTable(spark, silver_path):
        df_flat.write.format("delta") \
        .option("path", silver_path) \
        .saveAsTable("silver.user_recent_played")
        
        print("✅ Tabella creata e registrata con successo.")

   
   
   else:
       DeltaTable.forPath(spark, silver_path).alias('target').merge(
           df_flat.alias('source'),
           'target.played_at = source.played_at'
       ).whenNotMatchedInsert(
           values={
          "played_at":      "source.played_at",
          "track_id":       "source.track_id",
          "track_name":     "source.track_name",
          "first_artist_id":"source.first_artist_id",
          "album_id":       "source.album_id",
          "time_of_day":    "source.time_of_day",
          "ingested_ts":    "source.ingested_ts"
      }
  ).execute()

In [0]:
last_op = (
    spark.sql("Describe HISTORY my_spotify.silver.user_recent_played;")
    .select("version", "timestamp","operation", "operationMetrics")
    .orderBy("version", ascending=False)
    .first())

row_inserted = last_op['operationMetrics']["numTargetRowsInserted"]

output = {
    "status": "success",
    "run_date": run_date,
    "operation_type":last_op.operation,
    "rows_inserted": row_inserted
    }

# Return JSON to the Job
dbutils.notebook.exit(json.dumps(output))
